# 2. Multi-AMR Path Planning and Subcarrier Allocation
Generate seeded waypoint-based A* candidates, evaluate their sampled radio
rates, and compare NAIVE, +OPTBW, +SMARTPATH, and JOINT.

Run notebook 01 first to create `results/data/fingerprint.npz`. The planner
uses a static occupancy grid and does **not** enforce AMR-to-AMR collision
avoidance or continuous geometric clearance. See the planning assumptions in
README.md before interpreting the selected routes as robot trajectories.


## 0. Imports and Configuration

In [ ]:
# =========================================================
# Import
# =========================================================
import heapq
import importlib
import itertools

from math import hypot, sqrt

import matplotlib.pyplot as plt
import numpy as np

from matplotlib.patches import Circle, Rectangle


# =========================================================
# Load project_config.py
# =========================================================
import project_config
importlib.reload(project_config)

from project_config import (
    ROOM_X, ROOM_Y, BS_CENTER, RX_HEIGHT,
    N_SC, DF_HZ, P_REF_DBM, NOISE_SC_DBM,
    GRID_RES, N_ROWS, N_COLS, CELL_X, CELL_Y,
    OBSTACLES, R_QOS_REQUIRED_BPS,
    FINGERPRINT_PATH, PATH_PLANNING_RESULTS_PATH,
    DATA_DIR, FIGURES_DIR,
    AMR1_START_XY, AMR1_GOAL_XY, AMR2_START_XY, AMR2_GOAL_XY,
    AMR1_SPEED_MPS as AMR1_SPEED, AMR2_SPEED_MPS as AMR2_SPEED,
    AMR1_START_TIME_S as AMR1_START_TIME,
    AMR2_START_TIME_S as AMR2_START_TIME,
    OBSTACLE_BUFFER_M as OBSTACLE_BUFFER,
    SIM_DT_S as SIM_DT,
    N_PATH_CANDIDATES as K_CANDIDATES,
    MAX_PATH_RATIO, MAX_CANDIDATE_ATTEMPT_FACTOR,
    RANDOM_SEED, AMR1_LINK_OFFSET_DB, AMR2_LINK_OFFSET_DB,
)

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Planning grid:", (N_ROWS, N_COLS))
print("Grid resolution:", GRID_RES, "m")
print("Obstacle count:", len(OBSTACLES))

## Input: Load the Channel Fingerprint Produced in Step 1


In [ ]:
if not FINGERPRINT_PATH.exists():
    raise FileNotFoundError("Run notebook 01 first to generate results/data/fingerprint.npz.")

with np.load(FINGERPRINT_PATH, allow_pickle=False) as data:
    H = data["H"]
    snr_db = data["snr_db"]
    snr_sc_linear_map = data["snr_sc_linear"]

    CELL_X = data["cell_x"]
    CELL_Y = data["cell_y"]
    SC_RF_HZ = data["sc_rf_hz"]

    GRID_RES = float(data["grid_res"].item())
    ROOM_X = float(data["room_x"].item())
    ROOM_Y = float(data["room_y"].item())

# Reject grid/PHY mismatches instead of mixing a saved map with current settings.
expected_shape = (N_ROWS, N_COLS, N_SC)
if H.shape != expected_shape or snr_sc_linear_map.shape != expected_shape:
    raise ValueError("Fingerprint shape differs from project_config.py; rerun notebook 01.")
if snr_db.shape != expected_shape[:2]:
    raise ValueError("The average-SNR map has an invalid shape.")
for actual, expected in (
    (CELL_X, project_config.CELL_X), (CELL_Y, project_config.CELL_Y),
    (GRID_RES, project_config.GRID_RES),
    (ROOM_X, project_config.ROOM_X), (ROOM_Y, project_config.ROOM_Y),
):
    if np.shape(actual) != np.shape(expected) or not np.allclose(actual, expected):
        raise ValueError("Fingerprint grid differs from project_config.py; rerun notebook 01.")
expected_rf_hz = project_config.FREQ_HZ + (np.arange(N_SC) - N_SC // 2) * DF_HZ
if SC_RF_HZ.shape != (N_SC,) or not np.allclose(SC_RF_HZ, expected_rf_hz, rtol=0, atol=1):
    raise ValueError("Fingerprint frequencies differ from project_config.py; rerun notebook 01.")
if not np.all(np.isfinite(snr_sc_linear_map)) or np.any(snr_sc_linear_map < 0):
    raise ValueError("Fingerprint SNR must be finite and nonnegative.")

print(f"Fingerprint loaded: H={H.shape}")

## 1. Load Planning Parameters from `project_config.py`

In [ ]:
# Occupancy grid
MAX_CANDIDATE_ATTEMPTS = MAX_CANDIDATE_ATTEMPT_FACTOR * K_CANDIDATES

# Validate configuration values
if AMR1_SPEED <= 0 or AMR2_SPEED <= 0:
    raise ValueError("AMR speeds must be positive.")
if SIM_DT <= 0:
    raise ValueError("SIM_DT must be positive.")
if K_CANDIDATES < 1 or MAX_PATH_RATIO < 1.0:
    raise ValueError("Check the candidate-route configuration.")
if not np.all(np.isfinite([
    AMR1_SPEED, AMR2_SPEED, SIM_DT, AMR1_START_TIME, AMR2_START_TIME,
    AMR1_LINK_OFFSET_DB, AMR2_LINK_OFFSET_DB,
])):
    raise ValueError("Speeds, timing, and link offsets must be finite.")
if AMR1_START_TIME < 0 or AMR2_START_TIME < 0:
    raise ValueError("AMR start times must be nonnegative.")

# SNR is a power ratio: a dB offset becomes a multiplicative linear factor.
AMR1_LINK_SCALE = 10.0 ** (AMR1_LINK_OFFSET_DB / 10.0)
AMR2_LINK_SCALE = 10.0 ** (AMR2_LINK_OFFSET_DB / 10.0)


## 2. Build the Occupancy Grid

In [ ]:
def build_occupancy_grid(obstacle_buffer=OBSTACLE_BUFFER):
    """True: blocked; False: traversable"""

    # Construct grid-cell center coordinates
    grid_x, grid_y = np.meshgrid(CELL_X, CELL_Y)
    blocked = np.zeros((N_ROWS, N_COLS), dtype=bool)   # Initialize every cell as traversable

    for obstacle in OBSTACLES:
        center_x, center_y = obstacle["center"]
        kind = obstacle["kind"]

        # Mark cells inside a cylindrical obstacle as blocked
        if kind == "cylinder":
            distance = np.hypot(grid_x - center_x, grid_y - center_y)
            blocked |= distance <= obstacle["radius"] + obstacle_buffer

        # Mark cells inside a box obstacle as blocked
        elif kind == "box":
            size_x, size_y, _ = obstacle["size"]
            outside_x = np.maximum(np.abs(grid_x - center_x) - size_x/2, 0)
            outside_y = np.maximum(np.abs(grid_y - center_y) - size_y/2, 0)
            blocked |= np.hypot(outside_x, outside_y) <= obstacle_buffer

        else:
            raise ValueError(f"Unsupported obstacle type: {kind}")

    return blocked


blocked = build_occupancy_grid()
free_cells = np.argwhere(~blocked)

## 3. Coordinate and Grid-Cell Conversion Utilities

In [ ]:
# =========================================================
# Map a world-space (x, y) coordinate to its nearest grid cell.
# =========================================================
def world_to_cell(x, y):
    x, y = float(x), float(y)

    if not (0 <= x <= ROOM_X and 0 <= y <= ROOM_Y):
        raise ValueError(f"Coordinate lies outside the warehouse: ({x}, {y})")

    col = int(np.argmin(np.abs(CELL_X - x)))
    row = int(np.argmin(np.abs(CELL_Y - y)))
    return row, col


# =========================================================
# Map a grid cell to the world-space coordinates of its center.
# =========================================================
def cell_to_world(cell):
    row, col = map(int, cell)

    if not (0 <= row < N_ROWS and 0 <= col < N_COLS):
        raise ValueError(f"Cell lies outside the grid: {cell}")

    return float(CELL_X[col]), float(CELL_Y[row])


# =========================================================
# Convert a cell-based route into world-space (x, y) coordinates.
# =========================================================
def cells_to_xy(path):
    if path is None:
        return None
    return np.asarray([cell_to_world(cell) for cell in path], dtype=float)


# =========================================================
# Convert each AMR start and goal into grid-cell coordinates.
# =========================================================
AMR1_START_CELL = world_to_cell(*AMR1_START_XY)
AMR1_GOAL_CELL  = world_to_cell(*AMR1_GOAL_XY)
AMR2_START_CELL = world_to_cell(*AMR2_START_XY)
AMR2_GOAL_CELL  = world_to_cell(*AMR2_GOAL_XY)

AMR_ENDPOINTS = {
    "AMR1 start": AMR1_START_CELL,
    "AMR1 goal": AMR1_GOAL_CELL,
    "AMR2 start": AMR2_START_CELL,
    "AMR2 goal": AMR2_GOAL_CELL,
}

# Report the converted endpoints
for name, cell in AMR_ENDPOINTS.items():
    print(f"{name:10s}: cell={cell}, center={cell_to_world(cell)}")

## 4. A*

A*: a shortest-path search algorithm that avoids obstacles between the start and goal.\
\
f(n) = g(n) + h(n)\
\
n: current cell\
f: estimated total route cost\
g: accumulated route cost\
h: estimated remaining cost to the goal

In [ ]:
# Eight admissible moves: cardinal directions and diagonals
MOVES_8 = [
    (-1, 0, 1.0), (1, 0, 1.0), (0, -1, 1.0), (0, 1, 1.0),
    (-1, -1, sqrt(2)), (-1, 1, sqrt(2)),
    (1, -1, sqrt(2)), (1, 1, sqrt(2)),
]


# =========================================================
# Euclidean distance from the current cell to the goal
# =========================================================
def heuristic(cell, goal):
    r, c = cell
    rg, cg = goal
    return hypot(r - rg, c - cg)


# =========================================================
# Compute the physical route length in meters
# =========================================================
def path_length_m(path):
    if path is None:
        return np.inf

    return GRID_RES * sum(
        hypot(r2 - r1, c2 - c1)
        for (r1, c1), (r2, c2) in zip(path[:-1], path[1:])
    )


# =========================================================
# A* search routine
# =========================================================
def astar(blocked, start, goal):
    """
    path     : sequence of (row, col) cells from start to goal
    length_m : route length [m]
    """
    start, goal = tuple(start), tuple(goal)
    n_rows, n_cols = blocked.shape

    if blocked[start]:
        raise ValueError(f"Start is a blocked cell: {start}")
    if blocked[goal]:
        raise ValueError(f"Goal is a blocked cell: {goal}")

    counter = itertools.count()
    pq = [(heuristic(start, goal), next(counter), start)]
    g_score = {start: 0.0}
    came_from = {start: None}
    closed = set()

    while pq:
        _, _, current = heapq.heappop(pq)

        if current in closed:
            continue

        if current == goal:
            path = []
            while current is not None:
                path.append(current)
                current = came_from[current]

            path.reverse()
            return path, g_score[goal] * GRID_RES

        closed.add(current)
        r, c = current

        for dr, dc, move_cost in MOVES_8:
            nr, nc = r + dr, c + dc

            if not (0 <= nr < n_rows and 0 <= nc < n_cols):
                continue
            if blocked[nr, nc]:
                continue

            # Prevent diagonal moves from cutting through obstacle corners.
            if dr != 0 and dc != 0:
                if blocked[r, nc] or blocked[nr, c]:
                    continue

            neighbor = (nr, nc)
            new_g = g_score[current] + move_cost

            if new_g >= g_score.get(neighbor, np.inf):
                continue

            g_score[neighbor] = new_g
            came_from[neighbor] = current
            priority = new_g + heuristic(neighbor, goal)
            heapq.heappush(pq, (priority, next(counter), neighbor))

    return None, np.inf

print("A* search routine defined.")

## 5. Compute Shortest Routes for AMR1 and AMR2

In [ ]:
# Compute AMR1’s shortest route
AMR1_SHORTEST_PATH, AMR1_SHORTEST_LENGTH_M = astar(
    blocked, AMR1_START_CELL, AMR1_GOAL_CELL
)

# Compute AMR2’s shortest route
AMR2_SHORTEST_PATH, AMR2_SHORTEST_LENGTH_M = astar(
    blocked, AMR2_START_CELL, AMR2_GOAL_CELL
)

# Verify that both shortest-route searches succeeded.
if AMR1_SHORTEST_PATH is None or AMR2_SHORTEST_PATH is None:
    raise RuntimeError("Could not find a shortest route for one or more AMRs.")

# Convert cell-based routes to world-space (x, y) coordinates.
AMR1_SHORTEST_XY = cells_to_xy(AMR1_SHORTEST_PATH)
AMR2_SHORTEST_XY = cells_to_xy(AMR2_SHORTEST_PATH)


# Validate obstacle avoidance and route-length consistency.
for name, path, length_m in [
    ("AMR1", AMR1_SHORTEST_PATH, AMR1_SHORTEST_LENGTH_M),
    ("AMR2", AMR2_SHORTEST_PATH, AMR2_SHORTEST_LENGTH_M),
]:
    if any(blocked[cell] for cell in path):
        raise ValueError(f"{name}  route passes through a blocked cell.")

    if not np.isclose(path_length_m(path), length_m):
        raise ValueError(f"{name}  route-length calculation is inconsistent.")


# Summarize the shortest route for each AMR
for name, path, length_m in [
    ("AMR1", AMR1_SHORTEST_PATH, AMR1_SHORTEST_LENGTH_M),
    ("AMR2", AMR2_SHORTEST_PATH, AMR2_SHORTEST_LENGTH_M),
]:
    print(f"{name}: {len(path)} cells, {length_m:.2f} m")

In [ ]:
# Visualize the AMR1 and AMR2 shortest routes
import matplotlib.patheffects as pe

STROKE = [pe.withStroke(linewidth=4, foreground="black")]
fig, ax = plt.subplots(figsize=(11, 5.5))

# Average SNR map
im = ax.imshow(
    snr_db, origin="lower",
    extent=[0, ROOM_X, 0, ROOM_Y],
    cmap="turbo", aspect="equal",
)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Average SNR [dB]", fontsize=13)  # Label font size
cbar.ax.tick_params(labelsize=11)               # Tick-label font size

# Obstacles and their safety buffers
blocked_mask = np.ma.masked_where(~blocked, blocked.astype(float))
ax.imshow(
    blocked_mask, origin="lower",
    extent=[0, ROOM_X, 0, ROOM_Y],
    cmap="Greys", vmin=0, vmax=1,
    alpha=0.75, aspect="equal",
)

# A*-computed shortest routes
ax.plot(
    AMR1_SHORTEST_XY[:, 0], AMR1_SHORTEST_XY[:, 1],
    color="cyan", linestyle="-", linewidth=2.5,
    path_effects=STROKE,
    label=f"AMR1 shortest path ({AMR1_SHORTEST_LENGTH_M:.2f} m)",
    zorder=5,
)
ax.plot(
    AMR2_SHORTEST_XY[:, 0], AMR2_SHORTEST_XY[:, 1],
    color="orange", linestyle="--", linewidth=2.5,
    path_effects=STROKE,
    label=f"AMR2 shortest path ({AMR2_SHORTEST_LENGTH_M:.2f} m)",
    zorder=5,
)

# Start and goal markers
start_points = np.array([AMR1_SHORTEST_XY[0], AMR2_SHORTEST_XY[0]])
goal_points = np.array([AMR1_SHORTEST_XY[-1], AMR2_SHORTEST_XY[-1]])
ax.scatter(
    start_points[:, 0], start_points[:, 1],
    s=150, c="white", edgecolors="black",
    linewidths=1.5, marker="s", label="Start", zorder=6,
)
ax.scatter(
    goal_points[:, 0], goal_points[:, 1],
    s=170, c="white", edgecolors="black",
    linewidths=1.5, marker="^", label="Goal", zorder=6,
)
ax.scatter(
    *BS_CENTER[:2], s=340, c="white",
    edgecolors="black", linewidths=1.5,
    marker="*", label="BS", zorder=7,
)

# Configure and save the figure
ax.set(
    xlim=(0, ROOM_X), ylim=(0, ROOM_Y),
    xlabel="x [m]", ylabel="y [m]",
)
# Axis-label font size
ax.set_xlabel("x [m]", fontsize=13)
ax.set_ylabel("y [m]", fontsize=13)
ax.set_title("Shortest Paths", fontsize=15)
ax.grid(alpha=0.2)
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "02_shortest_paths.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Generate Candidate Routes

In [ ]:
# =========================================================
# Create a compact signature for a candidate route
# =========================================================
def path_signature(path):

    # Reject missing routes and routes with no recorded cells.
    if path is None or len(path) == 0:
        raise ValueError("Cannot create a signature for an empty route.")

    # Sample every third cell along the route.
    # Example: for path = [A, B, C, D, E, F, G, H], path[::3] = [A, D, G].
    sampled = tuple(tuple(cell) for cell in path[::3])

    # Retain the goal explicitly
    goal = tuple(path[-1])

    return sampled + (goal,)


# Inspect the shortest-route signatures
sig1 = path_signature(AMR1_SHORTEST_PATH)
sig2 = path_signature(AMR2_SHORTEST_PATH)

print("AMR1 signature:", sig1)
print("AMR2 signature:", sig2)

In [ ]:
# =========================================================
# Generate diverse candidate routes
# start ──A*──> random waypoint cell ──A*──> goal
# =========================================================
def generate_candidate_paths(
    blocked, start, goal, shortest_path, shortest_length_m,
    k=K_CANDIDATES, max_ratio=MAX_PATH_RATIO,
    max_attempts=MAX_CANDIDATE_ATTEMPTS,
):
    
    # Include the shortest route as the first candidate.
    candidates = [shortest_path]
    lengths_m = [shortest_length_m]

    # The shortest route occupies one slot; generate the remaining k - 1 alternatives.
    seen = {path_signature(shortest_path)}

    stats = {
        "attempts": 0,           # total sampled waypoint attempts
        "no_path": 0,            # attempts with no waypoint-to-goal route
        "too_long": 0,           # candidates rejected for exceeding the length limit
        "duplicate": 0,          # candidates rejected as near-duplicates
        "accepted_random": 0,    # randomly generated routes accepted as candidates
    }

    # Set the maximum allowable candidate-route length.
    length_limit_m = max_ratio * shortest_length_m

    # Sample waypoint cells up to max_attempts, stopping once k candidates are available.
    for _ in range(max_attempts):
        if len(candidates) >= k:
            break

        # Count every sampled waypoint.
        stats["attempts"] += 1

        # Randomly choose a traversable waypoint cell.
        via_idx = np.random.randint(len(free_cells))
        via = tuple(free_cells[via_idx])

        # start → via
        path_a, _ = astar(blocked, start, via)
        # via → goal
        path_b, _ = astar(blocked, via, goal)

        # Reject this waypoint if either subroute is unavailable.
        if path_a is None or path_b is None:
            stats["no_path"] += 1
            continue

        # Concatenate the start-to-waypoint and waypoint-to-goal routes.
        candidate = path_a + path_b[1:]

        # Measure the candidate-route length.
        length_m = path_length_m(candidate)

        # Reject routes that exceed the allowed multiple of the shortest-route length.
        if length_m > length_limit_m:
            stats["too_long"] += 1
            continue

        # Build a candidate signature using the helper defined above.
        signature = path_signature(candidate)

        # Reject candidates that duplicate an existing route signature.
        if signature in seen:
            stats["duplicate"] += 1
            continue

        # Record the accepted signature.
        seen.add(signature)
        candidates.append(candidate)
        lengths_m.append(length_m)
        stats["accepted_random"] += 1

    # Fail clearly if the requested candidate set cannot be constructed.
    if len(candidates) < k:
        raise RuntimeError(
            f"Could not generate {k} candidate routes. "
            f"Generated candidates={len(candidates)}, stats={stats}"
        )

    return candidates, np.asarray(lengths_m), stats


print("Candidate-route generator defined.")

## 7. Generate and Validate Candidate Routes per AMR


In [ ]:
# Fix the random seed so waypoint sampling is reproducible across runs.
np.random.seed(RANDOM_SEED)

# Generate AMR1 candidate routes
AMR1_CANDIDATE_PATHS, AMR1_CANDIDATE_LENGTHS, AMR1_CANDIDATE_STATS = generate_candidate_paths(
    blocked, AMR1_START_CELL, AMR1_GOAL_CELL,
    AMR1_SHORTEST_PATH, AMR1_SHORTEST_LENGTH_M
)

# Generate AMR2 candidate routes
AMR2_CANDIDATE_PATHS, AMR2_CANDIDATE_LENGTHS, AMR2_CANDIDATE_STATS = generate_candidate_paths(
    blocked, AMR2_START_CELL, AMR2_GOAL_CELL,
    AMR2_SHORTEST_PATH, AMR2_SHORTEST_LENGTH_M
)

# Convert generated candidate routes from (row, col) to (x, y).
AMR1_CANDIDATE_XY = [cells_to_xy(path) for path in AMR1_CANDIDATE_PATHS]
AMR2_CANDIDATE_XY = [cells_to_xy(path) for path in AMR2_CANDIDATE_PATHS]

# Validate the generated candidate sets
assert len(AMR1_CANDIDATE_PATHS) == K_CANDIDATES
assert len(AMR2_CANDIDATE_PATHS) == K_CANDIDATES
assert AMR1_CANDIDATE_LENGTHS.max() <= MAX_PATH_RATIO * AMR1_SHORTEST_LENGTH_M + 1e-9
assert AMR2_CANDIDATE_LENGTHS.max() <= MAX_PATH_RATIO * AMR2_SHORTEST_LENGTH_M + 1e-9

print(f"AMR1 candidate routes: {len(AMR1_CANDIDATE_PATHS)}, "
      f"length range {AMR1_CANDIDATE_LENGTHS.min():.2f}–{AMR1_CANDIDATE_LENGTHS.max():.2f} m")

print(f"AMR2 candidate routes: {len(AMR2_CANDIDATE_PATHS)}, "
      f"length range {AMR2_CANDIDATE_LENGTHS.min():.2f}–{AMR2_CANDIDATE_LENGTHS.max():.2f} m")

print("AMR1 generation statistics:", AMR1_CANDIDATE_STATS)
print("AMR2 generation statistics:", AMR2_CANDIDATE_STATS)

## 8. Convert Routes into Continuous Time-Indexed Positions

In [ ]:
# =========================================================
# Compute route travel time
# =========================================================
def path_duration_s(path_xy, speed):
    segment_lengths = np.linalg.norm(np.diff(path_xy, axis=0), axis=1)
    return segment_lengths.sum() / speed


# =========================================================
# Sample each AMR’s (x, y) position at specified times 
# =========================================================
def sample_path_positions(path_xy, speed, times, start_time=0.0):
    path_xy = np.asarray(path_xy, dtype=float)
    times = np.asarray(times, dtype=float)

    # A single-point route indicates no motion, so return that position at every time step.
    if len(path_xy) == 1:
        return np.repeat(path_xy, len(times), axis=0)

    # Compute the direction and length of every route segment.
    segment_vectors = np.diff(path_xy, axis=0)
    segment_lengths = np.linalg.norm(segment_vectors, axis=1)

    # Consecutive duplicate positions create a zero-length segment and would cause division by zero.
    if np.any(segment_lengths <= 0):
        raise ValueError("The route contains consecutive duplicate coordinates.")

    # Compute cumulative travel distance
    cumulative = np.concatenate(([0.0], np.cumsum(segment_lengths)))
    total_length = cumulative[-1]

    # Compute distance traveled by each time: distance = (t − start time) × speed.
    travelled = np.clip((times - start_time) * speed, 0.0, total_length)

    # Identify the active route segment.
    segment_idx = np.searchsorted(cumulative, travelled, side="right") - 1
    segment_idx = np.clip(segment_idx, 0, len(segment_lengths) - 1)

    # Compute the distance traveled within the active segment.
    local_distance = travelled - cumulative[segment_idx]

    # Express progress as a fraction of the active segment.
    ratio = local_distance / segment_lengths[segment_idx]

    # Interpolate position from the segment start, progress fraction, and segment vector.
    positions = (
        path_xy[segment_idx]
        + ratio[:, None] * segment_vectors[segment_idx]
    )

    # Keep the AMR at its final coordinate after arrival.
    positions[travelled >= total_length] = path_xy[-1]

    return positions


# =========================================================
# Build a common time axis through the actual mission-completion time 
# =========================================================
def build_time_axis(end_time):
    """
    Create a time axis from zero through the actual mission-completion time.

    Generate samples at intervals of SIM_DT,
    then append the exact mission-completion time as the final sample.
    """
    end_time = float(end_time)

    if end_time < 0:
        raise ValueError("end_time must be nonnegative.")

    # SIM_DT-spaced samples that do not exceed end_time
    times = np.arange(
        0.0,
        end_time,
        SIM_DT,
        dtype=float,
    )

    # Include the exact mission-completion time as the final sample.
    times = np.append(times, end_time)

    return times


# =========================================================
# Sanity-check the routine using the shortest-route pair
# =========================================================
test_end1 = (AMR1_START_TIME+ path_duration_s(AMR1_SHORTEST_XY, AMR1_SPEED))
test_end2 = (AMR2_START_TIME+ path_duration_s(AMR2_SHORTEST_XY, AMR2_SPEED))

test_end_time = max(test_end1, test_end2)

test_times = build_time_axis(test_end_time)

test_pos1 = sample_path_positions(
    AMR1_SHORTEST_XY, AMR1_SPEED, test_times, AMR1_START_TIME
)
test_pos2 = sample_path_positions(
    AMR2_SHORTEST_XY, AMR2_SPEED, test_times, AMR2_START_TIME
)

print(f"Common time axis: {test_times[0]:.1f}~{test_times[-1]:.1f} s")
print("Number of time samples:", len(test_times))
print(f"AMR1 travel time: {path_duration_s(AMR1_SHORTEST_XY, AMR1_SPEED):.2f} s")
print(f"AMR2 travel time: {path_duration_s(AMR2_SHORTEST_XY, AMR2_SPEED):.2f} s")
print("AMR1 initial/final position:", test_pos1[0], test_pos1[-1])
print("AMR2 initial/final position:", test_pos2[0], test_pos2[-1])

## 9. Interpolate Per-Subcarrier SNR at Each AMR Position

At the default speed of 0.5 m/s and sampling interval of 0.1 s, each AMR
travels 0.05 m per time step. The fingerprint grid spacing is 0.5 m.
Interpolate per-subcarrier SNR in linear units, then apply the configured
AMR link offsets (0 dB and −7 dB by default).


In [ ]:
# =========================================================
# Interpolate SNR values at the time-indexed positions computed in Step 8.
# =========================================================
def bilinear_sample_snr(positions_xy):
    positions_xy = np.asarray(positions_xy, dtype=float)

    # Validate and represent positions_xy as a floating-point array.
    if positions_xy.ndim != 2 or positions_xy.shape[1] != 2:
        raise ValueError("positions_xy must have shape (T, 2).")

    # Separate x and y coordinates.
    x, y = positions_xy[:, 0], positions_xy[:, 1]

    # Convert world coordinates to fractional grid indices.
    fx = (x - CELL_X[0]) / GRID_RES
    fy = (y - CELL_Y[0]) / GRID_RES

    # Find the lower-left grid cell for each position.
    col = np.clip(np.floor(fx).astype(int), 0, N_COLS - 2)
    row = np.clip(np.floor(fy).astype(int), 0, N_ROWS - 2)

    # Compute relative position within the enclosing grid cell.
    a = np.clip(fx - col, 0.0, 1.0)   # horizontal fraction from the left cell to the right cell
    b = np.clip(fy - row, 0.0, 1.0)   # vertical fraction from the lower cell to the upper cell

    # Retrieve per-subcarrier SNR from the four surrounding cells.
    snr00 = snr_sc_linear_map[row,     col,     :]
    snr10 = snr_sc_linear_map[row,     col + 1, :]
    snr01 = snr_sc_linear_map[row + 1, col,     :]
    snr11 = snr_sc_linear_map[row + 1, col + 1, :]

    # Bilinearly interpolate SNR from the four surrounding cells.
    return (
        (1-a)[:, None] * (1-b)[:, None] * snr00
        + a[:, None] * (1-b)[:, None] * snr10
        + (1-a)[:, None] * b[:, None] * snr01
        + a[:, None] * b[:, None] * snr11
    )


# =========================================================
# Apply the configured AMR-specific linear SNR scale.
# =========================================================
def sample_amr_snr(positions_xy, link_scale):
    return bilinear_sample_snr(positions_xy) * link_scale


# =========================================================
# Convert the mean SNR across 64 subcarriers to dB.
# =========================================================
def average_snr_db(snr_sc_linear):
    mean_linear = np.mean(snr_sc_linear, axis=1)
    return 10.0 * np.log10(np.maximum(mean_linear, 1e-30))


# =========================================================
# Sanity-check the routine using the shortest-route pair
# =========================================================
test_snr1_sc = sample_amr_snr(test_pos1, AMR1_LINK_SCALE)
test_snr2_sc = sample_amr_snr(test_pos2, AMR2_LINK_SCALE)

test_snr1_db = average_snr_db(test_snr1_sc)
test_snr2_db = average_snr_db(test_snr2_sc)

print("AMR1 per-SC SNR shape:", test_snr1_sc.shape)
print("AMR2 per-SC SNR shape:", test_snr2_sc.shape)

print(
    f"AMR1 mean SNR range: "
    f"{test_snr1_db.min():.2f}~{test_snr1_db.max():.2f} dB"
)

print(
    f"AMR2 mean SNR range: "
    f"{test_snr2_db.min():.2f}~{test_snr2_db.max():.2f} dB"
)

## 10. Allocate Subcarriers Between the Two AMRs

With the default 64 subcarriers, Equal-BW fixes a 32:32 split. Opt-BW tests
all 63 nonempty contiguous splits: AMR1 receives indices `[0, k)` and AMR2
receives `[k, N_SC)`. It maximizes the lower of the two Shannon rates at each
sample. This is optimal within that fixed-order contiguous-split family;
arbitrary subcarrier assignments, switching overhead, and interference are
not modeled. Transmit power remains fixed per active subcarrier.


In [ ]:
# =========================================================
# Sanity-check the routine using the shortest-route pair
# =========================================================

import simulator
importlib.reload(simulator)

from simulator import subcarrier_rate_bps, solve_opt_bw, solve_equal_bw


test_opt_bw = solve_opt_bw(test_snr1_sc, test_snr2_sc)
test_equal_bw = solve_equal_bw(test_snr1_sc, test_snr2_sc)

test_opt_rmin_mbps = test_opt_bw["min_rate_bps"].min() / 1e6
test_equal_rmin_mbps = test_equal_bw["min_rate_bps"].min() / 1e6


print(
    f"Opt-BW split range: "
    f"{test_opt_bw['split_k'].min()}~{test_opt_bw['split_k'].max()}"
)

print(f"Opt-BW minimum rate  : {test_opt_rmin_mbps:.2f} Mbps")
print(f"Equal-BW minimum rate: {test_equal_rmin_mbps:.2f} Mbps")

print(
    "Shortest-route pair satisfies QoS:",
    test_opt_bw["min_rate_bps"].min() >= R_QOS_REQUIRED_BPS
)

## 11. Evaluate One Route Pair

Both AMRs require service from time zero until the later arrival, including
waiting before departure and after arrival. QoS uses the minimum bottleneck
rate at the sampled times (default spacing 0.1 s, plus the exact final time);
it is not a guarantee between samples. `mean_rate_bps` is the arithmetic mean
of those samples, including the endpoint, rather than a time integral.


In [ ]:
# =========================================================
# Load simulator.py
# =========================================================
import simulator
importlib.reload(simulator)

from simulator import subcarrier_rate_bps, solve_opt_bw, solve_equal_bw

BW_SOLVERS = {
    "opt": solve_opt_bw,
    "equal": solve_equal_bw,
}


def evaluate_path_pair(
    path1_xy, path2_xy,
    bw_mode="opt",
    r_qos_bps=R_QOS_REQUIRED_BPS,
):
    if bw_mode not in BW_SOLVERS:
        raise ValueError(f"Unsupported bandwidth-allocation mode: {bw_mode}")

    path1_xy = np.asarray(path1_xy)
    path2_xy = np.asarray(path2_xy)

    # Route travel times
    duration1 = path_duration_s(path1_xy, AMR1_SPEED)
    duration2 = path_duration_s(path2_xy, AMR2_SPEED)

    # Arrival times
    arrival1 = AMR1_START_TIME + duration1
    arrival2 = AMR2_START_TIME + duration2

    # The later arrival time determines total mission completion.
    mission_time = max(arrival1, arrival2)

    # Build the common time axis using the Step 8 convention.
    times = build_time_axis(mission_time)

    # AMR positions over time
    pos1 = sample_path_positions(path1_xy, AMR1_SPEED, times, AMR1_START_TIME)
    pos2 = sample_path_positions(path2_xy, AMR2_SPEED, times, AMR2_START_TIME)

    # Compute the SNR of all 64 subcarriers over time.
    snr1_sc = sample_amr_snr(pos1, AMR1_LINK_SCALE)
    snr2_sc = sample_amr_snr(pos2, AMR2_LINK_SCALE)

    # Find the optimal contiguous subcarrier split at each time step.
    bw = BW_SOLVERS[bw_mode](snr1_sc, snr2_sc)

    # Locate the time of the minimum bottleneck rate.
    worst_idx = int(np.argmin(bw["min_rate_bps"]))
    r_min_bps = float(bw["min_rate_bps"][worst_idx])
    mean_rate_bps = float(bw["min_rate_bps"].mean())

    return {
        "bw_mode": bw_mode,
        "mission_time_s": float(mission_time),
        "arrival1_s": float(arrival1),
        "arrival2_s": float(arrival2),
        "duration1_s": float(duration1),
        "duration2_s": float(duration2),
        "time_s": times,
        "pos1_xy": pos1,
        "pos2_xy": pos2,
        "snr1_sc": snr1_sc,
        "snr2_sc": snr2_sc,
        "split_k": bw["split_k"],
        "rate1_bps": bw["rate1_bps"],
        "rate2_bps": bw["rate2_bps"],
        "min_rate_bps": bw["min_rate_bps"],
        "r_min_bps": r_min_bps,
        "mean_rate_bps": mean_rate_bps,
        "worst_idx": worst_idx,
        "worst_time_s": float(times[worst_idx]),
        "qos_feasible": bool(r_min_bps >= r_qos_bps),
    }


In [ ]:
# =========================================================
# Sanity-check using the shortest-route pair
# =========================================================
SHORTEST_EQUAL_RESULT = evaluate_path_pair(
    AMR1_SHORTEST_XY, AMR2_SHORTEST_XY, bw_mode="equal"
)

SHORTEST_OPT_RESULT = evaluate_path_pair(
    AMR1_SHORTEST_XY, AMR2_SHORTEST_XY, bw_mode="opt"
)

for name, res in [("Equal-BW", SHORTEST_EQUAL_RESULT), ("Opt-BW  ", SHORTEST_OPT_RESULT)]:
    k = res["split_k"]
    print(f"Shortest-route {name}: R_min={res['r_min_bps']/1e6:6.2f} Mbps, "
          f"mean={res['mean_rate_bps']/1e6:6.2f} Mbps, "
          f"split_k={k.min()}~{k.max()}, QoS={res['qos_feasible']}")

## 12. Evaluate All Candidate Route Pairs

The default 30 candidates per AMR produce 900 pairs. Select the minimum
mission time among QoS-feasible pairs, breaking ties by higher minimum rate.
If none meets QoS, return the highest-minimum-rate pair and report it as
infeasible. This search covers the generated candidate set, not all routes.


In [ ]:
# =========================================================
# Evaluate the Step 11 route-pair routine across all 900 combinations.
# =========================================================
def optimize_candidate_pairs(
    paths1, paths2, bw_mode="opt",
    r_qos_bps=R_QOS_REQUIRED_BPS, verbose=True,
):
    paths1_xy = [cells_to_xy(path) for path in paths1]
    paths2_xy = [cells_to_xy(path) for path in paths2]

    summaries = []
    best_feasible = None
    best_fallback = None
    total = len(paths1) * len(paths2)
    count = 0

    for i, path1_xy in enumerate(paths1_xy):
        for j, path2_xy in enumerate(paths2_xy):
            result = evaluate_path_pair(
                path1_xy, path2_xy,
                bw_mode=bw_mode,
                r_qos_bps=r_qos_bps,
            )

            T = result["mission_time_s"]
            R = result["r_min_bps"]
            summaries.append((i, j, T, R, result["qos_feasible"]))

            # Among QoS-feasible pairs, retain the one with the shortest mission time.
            feasible_key = (T, -R)
            if result["qos_feasible"] and (
                best_feasible is None or feasible_key < best_feasible[0]
            ):
                best_feasible = (feasible_key, i, j, result)

            # If no pair is QoS-feasible, retain the pair with the largest R_min.
            fallback_key = (-R, T)
            if best_fallback is None or fallback_key < best_fallback[0]:
                best_fallback = (fallback_key, i, j, result)

            count += 1
            if verbose and (count % 100 == 0 or count == total):
                print(f"\r{bw_mode} evaluation progress: {count}/{total}", end="")

    selected = best_feasible if best_feasible is not None else best_fallback
    mode = "minimum mission time among QoS-feasible routes" if best_feasible else "QoS infeasible → maximum R_min"

    _, best_i, best_j, best_result = selected
    return summaries, best_i, best_j, best_result, mode

## 13. Compare the Four Schemes

| Scheme | Route selection | Subcarrier allocation |
| --- | --- | --- |
| NAIVE | Shortest A* paths | Equal split |
| +OPTBW | Shortest A* paths | Optimized contiguous split |
| +SMARTPATH | Candidate-pair search | Equal split |
| JOINT | Candidate-pair search | Optimized contiguous split |


In [ ]:
# =========================================================
# Run the planner with equal bandwidth allocation → SMARTPATH
# =========================================================
SMART_PAIR_SUMMARIES, SMART_I, SMART_J, SMARTPATH_RESULT, SMARTPATH_MODE = (
    optimize_candidate_pairs(
        AMR1_CANDIDATE_PATHS,
        AMR2_CANDIDATE_PATHS,
        bw_mode="equal",
        verbose=False,
    )
)


# =========================================================
# Run the planner with optimized bandwidth allocation → JOINT
# =========================================================
PAIR_SUMMARIES, JOINT_I, JOINT_J, JOINT_RESULT, JOINT_MODE = (
    optimize_candidate_pairs(
        AMR1_CANDIDATE_PATHS,
        AMR2_CANDIDATE_PATHS,
        bw_mode="opt",
    )
)


# Routes selected by SMARTPATH
SMART_PATH1 = AMR1_CANDIDATE_PATHS[SMART_I]
SMART_PATH2 = AMR2_CANDIDATE_PATHS[SMART_J]
SMART_PATH1_XY = cells_to_xy(SMART_PATH1)
SMART_PATH2_XY = cells_to_xy(SMART_PATH2)


# Routes selected by JOINT
JOINT_PATH1 = AMR1_CANDIDATE_PATHS[JOINT_I]
JOINT_PATH2 = AMR2_CANDIDATE_PATHS[JOINT_J]
JOINT_PATH1_XY = cells_to_xy(JOINT_PATH1)
JOINT_PATH2_XY = cells_to_xy(JOINT_PATH2)


# =========================================================
# BEST denotes the JOINT result.
# =========================================================
BEST_I, BEST_J = JOINT_I, JOINT_J
BEST_RESULT = JOINT_RESULT

BEST_PATH1, BEST_PATH2 = JOINT_PATH1, JOINT_PATH2
BEST_PATH1_XY, BEST_PATH2_XY = JOINT_PATH1_XY, JOINT_PATH2_XY
SELECTION_MODE = JOINT_MODE


# Convert result lists to arrays and count QoS-feasible pairs.
PAIR_SUMMARIES = np.asarray(PAIR_SUMMARIES, dtype=float)
SMART_PAIR_SUMMARIES = np.asarray(SMART_PAIR_SUMMARIES, dtype=float)

N_FEASIBLE = int(PAIR_SUMMARIES[:, 4].sum())
SMART_N_FEASIBLE = int(SMART_PAIR_SUMMARIES[:, 4].sum())


# =========================================================
# Compare all four schemes
# =========================================================
SCHEMES = [
    ("NAIVE",      "Shortest ", "Equal", SHORTEST_EQUAL_RESULT, 0,       0),
    ("+OPTBW",     "Shortest ", "Opt  ", SHORTEST_OPT_RESULT,   0,       0),
    ("+SMARTPATH", "CommAware", "Equal", SMARTPATH_RESULT,      SMART_I, SMART_J),
    ("JOINT",      "CommAware", "Opt  ", JOINT_RESULT,          JOINT_I, JOINT_J),
]

print(f"\n\nQoS target: {R_QOS_REQUIRED_BPS/1e6:.0f} Mbps")
print(f"  QoS-feasible pairs with the Equal-BW planner: {SMART_N_FEASIBLE}/{len(SMART_PAIR_SUMMARIES)}"
      f"   ({SMARTPATH_MODE})")
print(f"  QoS-feasible pairs with the Opt-BW planner: {N_FEASIBLE}/{len(PAIR_SUMMARIES)}"
      f"   ({JOINT_MODE})")

print(f"\n{'scheme':<11} {'Path':<10} {'BW':<6} {'pair':<9}"
      f"{'R_min':>8} {'mean':>8} {'T':>8}   QoS")
print("-" * 72)

for name, path_kind, bw, res, i, j in SCHEMES:
    print(f"{name:<11} {path_kind:<10} {bw:<6} #{i+1:<3}#{j+1:<5}"
          f"{res['r_min_bps']/1e6:8.2f} {res['mean_rate_bps']/1e6:8.2f} "
          f"{res['mission_time_s']:8.2f}   {res['qos_feasible']}")

## 14. Visualize Routes for All Four Schemes

In [ ]:
import matplotlib.patheffects as pe

# Add black outlines so routes remain visible over the turbo colormap.
STROKE = [pe.withStroke(linewidth=3, foreground="black")]

SCHEME_PATHS = [
    ("NAIVE / +OPTBW  (shortest)",          "#e8e8e8",
     AMR1_SHORTEST_XY, AMR2_SHORTEST_XY),
    (f"+SMARTPATH  #{SMART_I+1},#{SMART_J+1}", "darkorange",
     SMART_PATH1_XY,   SMART_PATH2_XY),
    (f"JOINT  #{JOINT_I+1},#{JOINT_J+1}",      "red",
     JOINT_PATH1_XY,   JOINT_PATH2_XY),
]

fig, ax = plt.subplots(figsize=(11, 5.5))

# Average SNR map computed in Task 1
im = ax.imshow(snr_db, origin="lower", extent=[0, ROOM_X, 0, ROOM_Y],
               cmap="turbo", aspect="equal")
plt.colorbar(im, ax=ax, label="Average SNR [dB]")

# Obstacle and safety-buffer cells
blocked_mask = np.ma.masked_where(~blocked, blocked.astype(float))
ax.imshow(blocked_mask, origin="lower", extent=[0, ROOM_X, 0, ROOM_Y],
          cmap="Greys", vmin=0, vmax=1, alpha=0.75, aspect="equal")

# Scheme-specific routes (AMR1 solid / AMR2 dashed)
for label, color, p1, p2 in SCHEME_PATHS:
    ax.plot(p1[:, 0], p1[:, 1], "-",  color=color, lw=2, zorder=5,
            label=label, path_effects=STROKE)
    ax.plot(p2[:, 0], p2[:, 1], "--", color=color, lw=2, zorder=5,
            path_effects=STROKE)

# Square start markers / triangular goal markers
for xy, marker in [(AMR1_SHORTEST_XY[0],  "s"), (AMR1_SHORTEST_XY[-1], "^"),
                   (AMR2_SHORTEST_XY[0],  "s"), (AMR2_SHORTEST_XY[-1], "^")]:
    ax.scatter(*xy, s=150, c="white", edgecolors="black",
               linewidths=1.5, marker=marker, zorder=6)

ax.scatter(*BS_CENTER[:2], s=340, c="white", edgecolors="black",
           linewidths=1.5, marker="*", zorder=7, label="BS")

ax.set(xlim=(0, ROOM_X), ylim=(0, ROOM_Y), xlabel="x [m]", ylabel="y [m]")
ax.set_title(
    "Paths by scheme (solid: AMR1 / dashed: AMR2)"
)
ax.grid(alpha=0.2)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=4, fontsize=8)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "05_paths_by_scheme.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# =========================================================
# Render the routes for all four schemes in separate panels.
# =========================================================
import matplotlib.patheffects as pe

STROKE = [pe.withStroke(linewidth=3, foreground="black")]

PANELS = [
    ("NAIVE",      SHORTEST_EQUAL_RESULT, AMR1_SHORTEST_XY, AMR2_SHORTEST_XY, 0,       0),
    ("+OPTBW",     SHORTEST_OPT_RESULT,   AMR1_SHORTEST_XY, AMR2_SHORTEST_XY, 0,       0),
    ("+SMARTPATH", SMARTPATH_RESULT,      SMART_PATH1_XY,   SMART_PATH2_XY,   SMART_I, SMART_J),
    ("JOINT",      JOINT_RESULT,          JOINT_PATH1_XY,   JOINT_PATH2_XY,   JOINT_I, JOINT_J),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True, sharey=True)
blocked_mask = np.ma.masked_where(~blocked, blocked.astype(float))

for ax, (name, res, p1, p2, i, j) in zip(axes.ravel(), PANELS):
    im = ax.imshow(snr_db, origin="lower", extent=[0, ROOM_X, 0, ROOM_Y],
                   cmap="turbo", aspect="equal", vmin=snr_db.min(), vmax=snr_db.max())
    ax.imshow(blocked_mask, origin="lower", extent=[0, ROOM_X, 0, ROOM_Y],
              cmap="Greys", vmin=0, vmax=1, alpha=0.75, aspect="equal")

    # Routes (AMR1 solid / AMR2 dashed)
    ax.plot(p1[:, 0], p1[:, 1], "-",  color="cyan",   lw=2, zorder=5,
            path_effects=STROKE, label=f"AMR1 #{i+1}")
    ax.plot(p2[:, 0], p2[:, 1], "--", color="orange", lw=2, zorder=5,
            path_effects=STROKE, label=f"AMR2 #{j+1}")

    # Square start markers / triangular goal markers
    for xy, marker in [(p1[0], "s"), (p1[-1], "^"), (p2[0], "s"), (p2[-1], "^")]:
        ax.scatter(*xy, s=110, c="white", edgecolors="black",
                   linewidths=1.3, marker=marker, zorder=6)

    ax.scatter(*BS_CENTER[:2], s=260, c="white", edgecolors="black",
               linewidths=1.3, marker="*", zorder=7)

    # AMR positions at the R_min time instant
    w = res["worst_idx"]
    for pos in (res["pos1_xy"][w], res["pos2_xy"][w]):
        ax.scatter(*pos, s=240, facecolors="none", edgecolors="white",
                   linewidths=3.5, zorder=8)
        ax.scatter(*pos, s=240, facecolors="none", edgecolors="red",
                   linewidths=1.5, zorder=9)

    k = res["split_k"]
    ax.set_title(f"{name}   T={res['mission_time_s']:.2f} s, "
                 f"R_min={res['r_min_bps']/1e6:.2f} Mbps, "
                 f"k*={k.min()}~{k.max()}", fontsize=11)
    ax.set(xlim=(0, ROOM_X), ylim=(0, ROOM_Y))
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)

for ax in axes[-1]:
    ax.set_xlabel("x [m]")
for ax in axes[:, 0]:
    ax.set_ylabel("y [m]")

fig.colorbar(im, ax=axes, label="Average SNR [dB]", fraction=0.02, pad=0.02)
fig.suptitle("Paths by scheme (solid: AMR1, dashed: AMR2, circle: worst-rate instant)",
             fontsize=13)
fig.savefig(FIGURES_DIR / "06_paths_per_scheme.png", dpi=150, bbox_inches="tight")
plt.show()

## 15. Save Final Results


In [ ]:

SCHEME_RESULTS = {
    "naive": SHORTEST_EQUAL_RESULT,
    "optbw": SHORTEST_OPT_RESULT,
    "smartpath": SMARTPATH_RESULT,
    "joint": JOINT_RESULT,
}

SAVE_KEYS = (
    "time_s", "pos1_xy", "pos2_xy",
    "snr1_sc", "snr2_sc", "split_k",
    "rate1_bps", "rate2_bps", "min_rate_bps",
    "r_min_bps", "mean_rate_bps", "mission_time_s",
)

save_data = {}

for scheme, result in SCHEME_RESULTS.items():
    for key in SAVE_KEYS:
        save_data[f"{scheme}_{key}"] = result[key]

save_data.update({
    # QoS target and all optimized-BW route-pair evaluation results
    "r_qos_required_bps": R_QOS_REQUIRED_BPS,
    "pair_summaries": PAIR_SUMMARIES[:, :4],

    # Lengths of the generated candidate routes
    "amr1_candidate_lengths": np.asarray(
        AMR1_CANDIDATE_LENGTHS,
        dtype=float,
    ),
    "amr2_candidate_lengths": np.asarray(
        AMR2_CANDIDATE_LENGTHS,
        dtype=float,
    ),

    # Shortest-route coordinates and lengths
    "shortest_path1_xy": AMR1_SHORTEST_XY,
    "shortest_path2_xy": AMR2_SHORTEST_XY,
    "shortest_path1_length_m": AMR1_SHORTEST_LENGTH_M,
    "shortest_path2_length_m": AMR2_SHORTEST_LENGTH_M,

    # SMARTPATH-selected route coordinates and lengths
    "smartpath_path1_xy": SMART_PATH1_XY,
    "smartpath_path2_xy": SMART_PATH2_XY,
    "smartpath_path1_length_m": AMR1_CANDIDATE_LENGTHS[SMART_I],
    "smartpath_path2_length_m": AMR2_CANDIDATE_LENGTHS[SMART_J],

    # JOINT-selected route coordinates and lengths
    "joint_path1_xy": JOINT_PATH1_XY,
    "joint_path2_xy": JOINT_PATH2_XY,
    "joint_path1_length_m": AMR1_CANDIDATE_LENGTHS[JOINT_I],
    "joint_path2_length_m": AMR2_CANDIDATE_LENGTHS[JOINT_J],
})

np.savez_compressed(PATH_PLANNING_RESULTS_PATH, **save_data)

print("Path-planning results saved:", PATH_PLANNING_RESULTS_PATH.relative_to(project_config.PROJECT_DIR))
print("Number of saved items:", len(save_data))
print("Saved route-pair summaries:", save_data["pair_summaries"].shape)
print(
    "AMR1 candidate-route lengths:",
    save_data["amr1_candidate_lengths"].shape,
)
print(
    "AMR2 candidate-route lengths:",
    save_data["amr2_candidate_lengths"].shape,
)

## 16. Preview an Animation of the JOINT Routes

This cell embeds a JavaScript preview in Jupyter; it does not save a video.
Notebook 05 exports the MP4/GIF animations. The results are saved before this
preview, so rendering the animation is optional when only data are needed.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams["animation.embed_limit"] = 100.0
FRAME_STEP = 2

times = BEST_RESULT["time_s"]
pos1 = BEST_RESULT["pos1_xy"]
pos2 = BEST_RESULT["pos2_xy"]

frames = np.arange(0, len(times), FRAME_STEP, dtype=int)
if frames[-1] != len(times) - 1:
    frames = np.append(frames, len(times) - 1)

# Background layers
fig, ax = plt.subplots(figsize=(11, 5.5), constrained_layout=True)

im = ax.imshow(
    snr_db, origin="lower", extent=[0, ROOM_X, 0, ROOM_Y],
    cmap="turbo", aspect="equal"
)
plt.colorbar(
    im, ax=ax, label="Average SNR [dB]",
    fraction=0.03, pad=0.02, shrink=0.85
)

blocked_mask = np.ma.masked_where(~blocked, blocked.astype(float))
ax.imshow(
    blocked_mask, origin="lower", extent=[0, ROOM_X, 0, ROOM_Y],
    cmap="Greys", vmin=0, vmax=1, alpha=0.75, aspect="equal"
)

ax.scatter(
    BS_CENTER[0], BS_CENTER[1], s=170, c="red",
    edgecolors="white", marker="*", label="BS", zorder=7
)

# Full planned routes with black outlines
def draw_planned_path(path_xy, color):
    dash = (0, (6, 3))
    ax.plot(
        path_xy[:, 0], path_xy[:, 1], linestyle=dash,
        color="black", linewidth=4.0, alpha=0.9, zorder=4
    )
    ax.plot(
        path_xy[:, 0], path_xy[:, 1], linestyle=dash,
        color=color, linewidth=2.2, alpha=1.0, zorder=4.1
    )

draw_planned_path(BEST_PATH1_XY, "cyan")
draw_planned_path(BEST_PATH2_XY, "orange")

# Traversed routes and current AMR positions
amr1_trail, = ax.plot([], [], "-", color="cyan", linewidth=2.5, zorder=5)
amr2_trail, = ax.plot([], [], "-", color="orange", linewidth=2.5, zorder=5)

amr1_dot = ax.scatter(
    [], [], s=130, c="cyan", edgecolors="black",
    marker="o", label="AMR1", zorder=6
)
amr2_dot = ax.scatter(
    [], [], s=130, c="orange", edgecolors="black",
    marker="o", label="AMR2", zorder=6
)

status_text = ax.text(
    0.02, 0.96, "", transform=ax.transAxes,
    ha="left", va="top", fontsize=9,
    bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "black"},
    zorder=8
)

ax.set(
    xlim=(0, ROOM_X), ylim=(0, ROOM_Y),
    xlabel="x [m]", ylabel="y [m]"
)
ax.set_title("Selected AMR Path Animation", pad=4)
ax.grid(alpha=0.2)
ax.legend(loc="upper right", fontsize=9)

# Update one animation frame
def update(frame_idx):
    amr1_dot.set_offsets(pos1[frame_idx][None, :])
    amr2_dot.set_offsets(pos2[frame_idx][None, :])

    amr1_trail.set_data(pos1[:frame_idx + 1, 0], pos1[:frame_idx + 1, 1])
    amr2_trail.set_data(pos2[:frame_idx + 1, 0], pos2[:frame_idx + 1, 1])

    current_time = times[frame_idx]
    rate1_mbps = BEST_RESULT["rate1_bps"][frame_idx] / 1e6
    rate2_mbps = BEST_RESULT["rate2_bps"][frame_idx] / 1e6
    split_k = BEST_RESULT["split_k"][frame_idx]

    status_text.set_text(
        f"t={current_time:.1f} s  |  "
        f"AMR1={rate1_mbps:.1f} Mbps  |  "
        f"AMR2={rate2_mbps:.1f} Mbps  |  "
        f"BW={split_k}:{N_SC-split_k}"
    )

    return amr1_dot, amr2_dot, amr1_trail, amr2_trail, status_text

animation = FuncAnimation(
    fig, update, frames=frames,
    interval=SIM_DT * FRAME_STEP * 1000,
    blit=True, repeat=False
)

plt.close(fig)
HTML(animation.to_jshtml(default_mode="once"))